# 사내 규정 챗봇 만들기



## RAG 실습 준비

In [ ]:
# 사내 규정 챗봇 만들기 - RAG 실습
# 이 코드는 실제 RAG 시스템을 구축하는 전 과정을 단계별로 보여줍니다

# 필요한 라이브러리 설치
# - langchain_community: 로더 등 커뮤니티 제공 컴포넌트
# - langchain_text_splitters: 문서 분할 도구
# - langchain_openai: OpenAI 모델 및 임베딩
# - langchain_chroma: 벡터 데이터베이스(Chroma) 통합
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 75.3 MB/s eta 0:00:00
 

In [ ]:
# 라이브러리 임포트
import os  # 운영체제 관련 기능 (환경변수, 파일 경로 등)

# LangChain 컴포넌트들 임포트
from langchain_community.document_loaders import TextLoader  # 텍스트 파일 로더
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 재귀적 문서 분할기
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI 임베딩과 채팅 모델
from langchain_chroma import Chroma  # 벡터 데이터베이스
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿
from langchain_core.output_parsers import StrOutputParser  # 문자열 출력 파서
from langchain_core.runnables import RunnablePassthrough  # 입력 데이터를 그대로 전달하는 래퍼

# 이 임포트 구문들의 구체적 역할:

# 1. TextLoader:
#    - 로컬 파일 시스템이나 URL에서 텍스트 문서를 읽어옴
#    - 다양한 포맷(txt, pdf, docx 등) 지원하는 로더들이 있음
#    - 여기서는 단순 텍스트 파일을 로드하기 위해 사용

# 2. RecursiveCharacterTextSplitter:
#    - 문서를 의미 단위로 재귀적으로 분할하는 클래스
#    - 문단, 문장, 단어 단위로 순차적으로 분할을 시도
#    - chunk_size와 chunk_overlap 파라미터로 분할 크기 조정

# 3. OpenAIEmbeddings:
#    - OpenAI의 임베딩 모델(text-embedding-ada-002 등)을 사용
#    - 텍스트를 고차원 벡터(숫자 배열)로 변환
#    - 벡터 간 유사도 계산을 통해 의미적으로 유사한 문서 찾기

# 4. ChatOpenAI:
#    - OpenAI의 채팅 모델(GPT-3.5, GPT-4 등)을 사용
#    - 질문에 대한 답변 생성

# 5. Chroma:
#    - 오픈소스 벡터 데이터베이스
#    - 임베딩 벡터를 저장하고 유사도 검색 수행
#    - 메모리 기반 또는 디스크 지속성 지원

# 6. PromptTemplate:
#    - 프롬프트 작성을 위한 템플릿 시스템
#    - 변수 치환을 통해 동적 프롬프트 생성

# 7. StrOutputParser:
#    - LLM의 출력을 문자열로 파싱
#    - 구조화된 응답을 평문으로 변환

# 8. RunnablePassthrough:
#    - LCEL(체인 표현 언어)에서 사용하는 특수 컴포넌트
#    - 입력값을 변경 없이 다음 단계로 전달
#    - 체인에서 특정 데이터를 그대로 유지하고 싶을 때 사용

# 이 코드가 실습할 RAG 시스템의 전체 아키텍처:
# 1. 문서 로딩(Loading) → 2. 문서 분할(Splitting) → 3. 임베딩 & 인덱싱(Indexing) →
# 4. 검색(Retrieval) → 5. 생성(Generation)

# 각 라이브러리가 RAG 파이프라인의 어느 부분에 해당하는지:
# - TextLoader: 문서 로딩 단계
# - RecursiveCharacterTextSplitter: 문서 분할 단계
# - OpenAIEmbeddings & Chroma: 임베딩 & 인덱싱 단계
# - Retriever(Chroma 기반): 검색 단계
# - ChatOpenAI & PromptTemplate: 생성 단계

# 실제 프로덕션 환경과의 차이점:
# 1. 더 다양한 문서 포맷 지원 (PDFLoader, DocxLoader 등)
# 2. 분할 전처리 (표, 이미지, 수식 등 특수 요소 처리)
# 3. 임베딩 모델 선택 (OpenAI 외 Cohere, HuggingFace 등)
# 4. 벡터 DB 선택 (Chroma 외 Pinecone, Weaviate, Qdrant 등)
# 5. 복잡한 프롬프트 엔지니어링 (Few-shot, Chain-of-Thought 등)

# 디버깅과 모니터링을 위한 추가 라이브러리:
# - langchain.callbacks: 체인 실행 과정 모니터링
# - langchain.evaluation: RAG 시스템 성능 평가
# - tiktoken: 토큰 수 계산 및 비용 관리

# 이 코드가 실행되는 환경 요구사항:
# 1. Python 3.8 이상
# 2. OpenAI API 키 (유효한 결제 계정 필요)
# 3. 충분한 메모리 (벡터 DB와 문서 임베딩 저장용)
# 4. 인터넷 연결 (OpenAI API 호출용)

# 다음 단계에서 수행할 작업:
# 1. API 키 설정 및 가상 데이터 생성
# 2. 실제 문서 로딩 및 전처리
# 3. 벡터 데이터베이스 구축
# 4. RAG 체인 구성 및 테스트

In [ ]:
# ==========================================
# 0. 초기 설정 (Initial Setup)
# ==========================================
# 이 부분은 RAG 시스템을 실행하기 위한 필수 환경 설정을 담당합니다
# API 키 설정, 환경변수 구성 등 시스템의 기본적인 준비 작업을 수행합니다

# getpass 모듈에서 getpass 함수 임포트
# getpass는 사용자로부터 비밀번호 같은 민감한 정보를 입력받을 때 사용됩니다
# 콘솔 환경에서 입력값이 화면에 보이지 않도록 해줍니다
from getpass import getpass

# OpenAI API 키 입력 받기
# getpass() 함수는 터미널에서 안전하게 입력을 받을 수 있게 해줍니다:
# - 일반 input()과 달리 입력한 내용이 화면에 표시되지 않음
# - Colab이나 Jupyter에서는 별도의 팝업 창이 나타남
# - 실제 키: sk-로 시작하는 긴 문자열 (OpenAI 계정에서 발급받음)
openai_api_key = getpass("Enter your OpenAI API key: ")

# API 키를 환경변수에 설정
# os.environ은 현재 프로세스의 환경변수를 관리하는 딕셔너리와 같은 객체입니다
# "OPENAI_API_KEY"라는 이름으로 API 키를 저장하면:
# 1. LangChain의 OpenAI 관련 클래스들이 자동으로 이 키를 인식
# 2. API 호출 시마다 키를 명시적으로 전달할 필요 없음
# 3. 코드에 키를 하드코딩하지 않아 보안성이 향상됨
os.environ["OPENAI_API_KEY"] = openai_api_key

# 이 코드의 구체적 실행 과정:
#
# [사용자 입력 단계]
# 1. getpass("Enter your OpenAI API key: ")가 실행되면:
#    - 콘솔에 "Enter your OpenAI API key: " 메시지 출력
#    - 사용자 입력 대기 상태로 전환
#    - 사용자가 키를 입력하는 동안 화면에 아무것도 표시되지 않음
#    - 입력 완료 후(Enter 키) 입력값이 openai_api_key 변수에 저장됨
#
# [환경변수 설정 단계]
# 2. os.environ["OPENAI_API_KEY"] = openai_api_key 실행:
#    - 현재 Python 프로세스의 환경변수 딕셔너리에 키-값 쌍 추가
#    - 키: "OPENAI_API_KEY", 값: 사용자가 입력한 API 키 문자열
#    - 이 설정은 현재 실행 중인 Python 프로세스 내에서만 유효
#    - 프로그램을 재실행하거나 다른 터미널에서 실행하면 설정이 사라짐

# 왜 환경변수를 사용하는가?
# 1. 보안: 코드에 API 키를 직접 작성하지 않아 코드 유출 시 키도 함께 유출되는 문제 방지
# 2. 유연성: 개발/테스트/운영 환경마다 다른 키를 사용할 수 있음
# 3. 표준화: 많은 라이브러리들이 환경변수를 통해 설정값을 읽는 방식으로 설계됨

# 실제 프로덕션 환경에서의 API 키 관리:
# 1. 환경변수 파일(.env) 사용: python-dotenv 같은 라이브러리로 관리
# 2. 시크릿 관리 서비스: AWS Secrets Manager, HashiCorp Vault 등
# 3. CI/CD 파이프라인: GitHub Actions의 Secrets, GitLab CI 변수 등
# 4. 컨테이너 환경: Docker의 환경변수, Kubernetes의 ConfigMap/Secret

# 대안적 API 키 설정 방법들:
# 1. 직접 코드에 작성 (비추천, 보안 취약):
#    os.environ["OPENAI_API_KEY"] = "sk-..."
#
# 2. 환경변수 파일 사용 (권장):
#    # .env 파일에 OPENAI_API_KEY=sk-... 작성
#    from dotenv import load_dotenv
#    load_dotenv()
#
# 3. 명령줄에서 설정:
#    # 터미널에서: export OPENAI_API_KEY="sk-..."
#    # 그 후 Python 실행

# API 키 유효성 검증:
# 여기서는 입력받은 키의 유효성을 검사하지 않습니다
# 실제로는 API 키 테스트 호출을 통해 유효한지 확인하는 로직을 추가할 수 있음

# 오류 시나리오 및 처리:
# 1. 사용자가 잘못된 키 입력: 후속 API 호출에서 401 에러 발생
# 2. 키 미입력: 빈 문자열이 설정되면 API 호출 실패
# 3. 키 형식 오류: sk-로 시작하지 않으면 인증 실패

# 시스템 준비 상태 확인:
# 이 설정이 완료되면 이후 코드에서:
# - OpenAIEmbeddings: API 키를 자동으로 읽어 임베딩 생성 가능
# - ChatOpenAI: API 키를 자동으로 읽어 LLM 호출 가능
# - 기타 OpenAI 기반 컴포넌트들: 모두 동일한 키 사용

# 참고: 다른 LLM 제공업체 사용 시
# - Anthropic: os.environ["ANTHROPIC_API_KEY"] = ...
# - Google AI: os.environ["GOOGLE_API_KEY"] = ...
# - HuggingFace: os.environ["HF_TOKEN"] = ...
# 각 제공업체마다 환경변수 이름이 다를 수 있음

# 보안 모범 사례:
# 1. API 키는 절대 공개 저장소(GitHub 등)에 커밋하지 않음
# 2 .env 파일은 .gitignore에 추가하여 버전 관리에서 제외
# 3. API 키는 정기적으로 교체 (OpenAI 대시보드에서 재생성 가능)
# 4. 최소 권한 원칙: 필요한 기능만 허용하는 키 사용

# 다음 단계에서 이 설정이 어떻게 활용되는지:
# 1. OpenAIEmbeddings 객체 생성 시 자동으로 os.environ["OPENAI_API_KEY"]를 읽음
# 2. ChatOpenAI 객체 생성 시 동일한 방식으로 키를 사용
# 3. 모든 API 호출은 이 키를 사용하여 인증됨

# 이 초기 설정의 중요성:
# RAG 시스템의 모든 구성요소(임베딩, LLM, 검색기 등)가 정상 작동하려면
# 올바른 API 키 설정이 필수적입니다. 이는 시스템의 출발점 역할을 합니다.

Enter your OpenAI API key: ··········


In [ ]:
# 실습용 가상 데이터 생성 (사내 규정 문서)
# 실제 RAG 시스템에서는 기존의 PDF, Word, 텍스트 파일 등을 사용하지만,
# 실습 편의를 위해 Python 코드 내에서 직접 텍스트를 생성하고 파일로 저장합니다.

# 사내 규정 내용을 담은 문자열 변수 정의
# 이 텍스트는 RAG 시스템이 참고할 지식베이스(knowledge base) 역할을 합니다
policy_content = """
[주식회사 랭체인 인사 규정]
제1조 (근무 시간)
1. 근무 시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연 근무제를 시행하며, 코어 타임(오전 11시~오후 3시)을 준수해야 한다.

제2조 (휴가)
1. 연차 휴가는 입사 1년 후 15일이 발생한다.
2. 3년 이상 근속 시 안식월 1개월을 유급으로 제공한다.
3. 반차, 반반차 사용이 가능하며 당일 신청도 가능하다.
4. 여름 휴가는 7월~8월 중 5일을 별도로 제공한다.

제3조 (복지)
1. 도서 구입비는 월 10만원 한도 내에서 무제한 지원한다.
2. 야근 시 택시비와 식대는 법인 카드로 결제한다.
"""

# 파일 저장 작업
# with 문을 사용하여 파일을 안전하게 열고 닫습니다
# open() 함수 매개변수:
# - "company_policy.txt": 저장할 파일 이름
# - "w": 쓰기 모드 (파일이 없으면 생성, 있으면 덮어씀)
# - encoding="utf-8": UTF-8 인코딩으로 한글 포함 텍스트 저장
with open("company_policy.txt", "w", encoding="utf-8") as f:
    # 파일 객체 f에 policy_content 문자열을 작성
    f.write(policy_content)

# 완료 메시지 출력
print("[준비 완료] 사내 규정 문서가 생성되었습니다.")

# 파일 저장 후의 상태:
# - 현재 작업 디렉토리에 "company_policy.txt" 파일 생성됨
# - 파일 크기: 약 400-500바이트 (한글 1글자 약 3바이트)
# - 파일 내용은 policy_content 변수와 동일

# 이 코드가 RAG 파이프라인에서 수행하는 역할:
# 1. 지식베이스 생성: RAG 시스템이 참조할 문서 소스 생성
# 2. 데이터 준비: 문서 로딩 단계에서 사용할 실제 파일 생성
# 3. 실습 환경 구축: 실제 파일 I/O 작업을 시뮬레이션

# 실제 프로덕션 환경과의 차이점:
# 1. 실제: 기존에 존재하는 문서 파일들(HR 매뉴얼, 정책 PDF 등) 사용
# 2. 실제: 다양한 포맷(PDF, DOCX, HTML, PPT 등)의 문서 처리 필요
# 3. 실제: 문서 용량이 훨씬 크고 구조가 복잡함 (수십~수백 페이지)
# 4. 실제: 여러 문서를 하나의 지식베이스로 통합

# 텍스트 구조화의 중요성:
# - 제목, 조항, 항목으로 구조화되어 있어 LLM이 이해하기 쉬움
# - 실제 문서도 적절한 구조화가 검색 정확도에 중요함

# 파일 저장 시 고려사항:
# 1. 인코딩: 한글 등 비영어 텍스트는 UTF-8 사용이 필수
# 2. 경로: 상대경로(현재 디렉토리) 또는 절대경로 지정 가능
# 3. 파일명: 중복되지 않는 고유한 이름 사용

# 실습용 데이터 설계 원리:
# 1. 다양성: 근무 시간, 휴가, 복지 등 다양한 주제 포함
# 2. 구체성: 숫자(시간, 금액, 기간)와 조건 포함
# 3. 실제성: 실제 회사 규정과 유사한 내용 구성
# 4. 검색 가능성: 키워드(안식월, 코어 타임, 법인 카드 등) 포함

# 파일 생성 후의 다음 단계:
# 1. TextLoader를 사용하여 방금 생성한 파일을 로드
# 2. 문서를 분석 가능한 형태(Document 객체)로 변환
# 3. 이후 분할, 임베딩, 검색 단계로 진행

# 오류 가능성 및 처리:
# 1. 권한 부족: 디렉토리에 쓰기 권한이 없으면 파일 생성 실패
# 2. 디스크 공간 부족: 매우 큰 파일을 생성할 때 발생 가능
# 3. 인코딩 오류: 다른 인코딩으로 저장하면 한글 깨짐

# 실제 시스템에서의 문서 관리:
# 1. 버전 관리: 정책 변경 시 문서 버전 관리 필요
# 2. 업데이트 주기: 주기적으로 문서 최신화
# 3. 접근 권한: 민감한 문서는 접근 제어 필요

# 이 단계의 중요성:
# RAG 시스템의 성능은 지식베이스의 질에 크게 의존합니다.
# 정확하고 구조화된 문서는 더 정확한 검색과 답변을 가능하게 합니다.

[준비 완료] 사내 규정 문서가 생성되었습니다.


## RAG 실습 시작 - LangChain 사용

### 문서 로딩 (Loading)

In [ ]:
# ==========================================
# 1. 문서 로딩 (Loading)
# ==========================================
# RAG 파이프라인의 첫 번째 단계: 외부 문서를 시스템으로 불러오기
# 이 단계에서는 파일 시스템, 웹, 데이터베이스 등 다양한 소스에서 문서를 로드합니다

print("\n 문서를 불러옵니다...")

# TextLoader 객체 생성
# - 첫 번째 인자: "company_policy.txt" - 로드할 파일 경로
# - 두 번째 인자: encoding="utf-8" - 파일 인코딩 방식 (한글 처리 필수)
loader = TextLoader("company_policy.txt", encoding="utf-8")

# loader.load() 메서드 실행: 실제 파일 로딩 작업 수행
# 반환값: Document 객체들의 리스트 (여기서는 하나의 문서만 로드하므로 리스트 길이 1)
docs = loader.load()

# 로드된 문서 정보 출력
# docs[0]: 첫 번째 Document 객체
# .page_content: Document 객체의 실제 텍스트 내용
# len(): 문자열의 길이(문자 수)를 계산
print(f"   -> 문서 길이: {len(docs[0].page_content)}자")

# TextLoader의 내부 동작 과정:
# 1. 파일 열기: 지정된 경로의 파일을 읽기 모드로 열기
# 2. 내용 읽기: 파일 전체 내용을 문자열로 읽어들임
# 3. Document 객체 생성:
#    - page_content: 파일 내용 문자열
#    - metadata: 파일 경로, 소스 정보 등 메타데이터 포함
# 4. 리스트로 감싸서 반환

# Document 객체의 구조:
# Document(
#   page_content="[주식회사 랭체인 인사 규정]\n제1조 (근무 시간)...",
#   metadata={'source': 'company_policy.txt'}
# )

# 실제 다양한 로더들:
# 1. PDFLoader: PDF 파일에서 텍스트 추출 (PyPDF2, pdfminer 등 사용)
# 2. DocxLoader: Microsoft Word 문서 처리
# 3. WebBaseLoader: 웹페이지에서 내용 스크래핑
# 4. CSVLoader: CSV 파일에서 테이블 데이터 로드
# 5. DirectoryLoader: 디렉토리 내 모든 파일 일괄 로드

# 로딩 과정에서 발생할 수 있는 문제들:
# 1. 파일 없음: 지정된 경로에 파일이 없으면 FileNotFoundError 발생
# 2. 인코딩 오류: 지정된 인코딩과 실제 파일 인코딩이 다르면 읽기 실패
# 3. 메모리 부족: 매우 큰 파일을 한 번에 읽으려고 하면 메모리 오류
# 4. 권한 문제: 파일 읽기 권한이 없으면 접근 거부

# 로드된 문서의 전처리 필요성:
# 1. 불필요한 공백 제거
# 2. 특수 문자 정리
# 3. 인코딩 문제 해결
# 4. 구조화된 데이터 추출 (표, 목록 등)

# LangChain의 로더 시스템 장점:
# 1. 통일된 인터페이스: 모든 로더가 load() 메서드 제공
# 2. 메타데이터 자동 추출: 소스, 생성 시간 등 정보 보존
# 3. 확장성: 사용자 정의 로더 쉽게 생성 가능
# 4. 다양한 소스 지원: 로컬 파일, 웹, 클라우드 스토리지 등

# 대용량 문서 처리 방법:
# 1. 청크 단위로 읽기: 전체를 한 번에 읽지 않고 부분적으로 처리
# 2. 병렬 로딩: 여러 파일을 동시에 로드
# 3. 지연 로딩: 필요한 시점에만 문서 내용 로드

# 디버깅을 위한 체크포인트:
# 1. 파일 경로 확인: 현재 작업 디렉토리 위치 확인
# 2. 파일 내용 검증: 문서 내용이 제대로 로드되었는지 확인
# 3. 메타데이터 확인: 소스 정보 등 메타데이터 포함 여부

# 다음 단계(분할)를 위한 준비:
# - 현재 문서는 하나의 긴 문자열로 되어 있음
# - LLM의 컨텍스트 창 크기에 맞게 분할 필요
# - 의미 단위로 분할하여 검색 정확도 향상

# 실시간 처리와 배치 처리:
# 1. 실시간: 사용자 질문 시마다 문서 로드 (느림)
# 2. 배치: 사전에 문서를 로드하여 벡터 DB에 저장 (빠름, 이 예제의 방식)

# 이 단계의 출력 결과:
# 문서를 불러옵니다...
#    -> 문서 길이: XXX자
#
# 실제 길이는 policy_content의 문자 수에 따라 달라집니다.

# 성능 최적화 팁:
# 1. 캐싱: 동일 문서 반복 로드 방지
# 2. 증분 로딩: 변경된 부분만 업데이트
# 3. 압축 해제: 압축된 문서 실시간 해제

# 보안 고려사항:
# 1. 신뢰할 수 없는 소스의 문서는 샌드박스 환경에서 로드
# 2. 악성 코드가 포함된 파일 처리 주의
# 3. 개인정보 포함 문서는 익명화 처리

# 이 로딩 단계가 RAG 시스템 전체에 미치는 영향:
# 로딩된 문서의 질이 검색과 생성 단계의 성능을 결정짓는 중요한 요소입니다.
# 정확하고 완전한 문서 로딩은 신뢰할 수 있는 답변의 기초가 됩니다.


 문서를 불러옵니다...
   -> 문서 길이: 321자


### 문서 분할 (Splitting)

In [ ]:
# ==========================================
# 2. 문서 분할 (Splitting)
# ==========================================
# RAG 파이프라인의 두 번째 단계: 긴 문서를 작은 단위로 분할
# LLM의 컨텍스트 제한과 검색 정확도를 위해 문서를 청크(조각)로 나눕니다

print("\n 문서를 작은 조각(Chunk)으로 나눕니다...")

# RecursiveCharacterTextSplitter 객체 생성
# 이 분할기는 재귀적 알고리즘을 사용하여 의미 단위로 문서를 분할합니다
#
# 매개변수 설명:
# - chunk_size=100: 각 청크의 목표 크기 (문자 단위)
#   실제로는 이 크기를 최대한 유지하려 하지만, 의미 단위를 우선시할 수 있음
# - chunk_overlap=20: 인접한 청크 간 겹치는 문자 수
#   문맥 정보가 끊기지 않도록 하여 검색 품질을 높입니다
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 문서 분할 실행
# split_documents() 메서드는 Document 객체 리스트를 입력받아 분할된 Document 객체 리스트를 반환
# docs는 이전 단계에서 로드한 하나의 Document를 포함한 리스트
splits = text_splitter.split_documents(docs)

# 분할 결과 출력
# len(splits): 분할된 청크의 개수
print(f"   -> 총 {len(splits)}개의 조각(Chunk)으로 분할되었습니다.")

# RecursiveCharacterTextSplitter의 내부 작동 알고리즘:
# 1. 분할 기준 설정: 기본적으로 ["\n\n", "\n", " ", ""] 순서로 분할 시도
#    (문단 → 줄 → 단어 → 문자 순으로 점점 작은 단위로 재귀적 분할)
# 2. chunk_size 확인: 현재 청크가 지정된 크기를 초과하면 다음 분할기준으로 분할 시도
# 3. chunk_overlap 적용: 이전 청크의 끝 부분을 다음 청크 시작에 포함

# 분할 전과 후의 문서 구조 변화:
#
# [분할 전]
# Document 1: 전체 규정 텍스트 (약 400-500자)
#
# [분할 후]
# Document 1: "[주식회사...] ~ 코어 타임(오전 11시~오후 3시)을..." (첫 ~100자)
# Document 2: "코어 타임(오전 11시~오후 3시)을 ~ 15일이 발생한다." (겹침 20자 포함)
# Document 3: "3년 이상 근속 시 ~ ..." (계속)
# ...

# 분할이 필요한 이유:
# 1. LLM 컨텍스트 제한: 대부분 LLM은 입력 토큰 수에 제한이 있음 (예: GPT-4o-mini는 128K)
# 2. 검색 정확도: 작은 청크 단위로 검색하면 질문과 더 관련성 높은 내용 찾기 쉬움
# 3. 효율성: 관련 없는 긴 내용을 전체적으로 처리하는 것보다 핵심 내용만 전달 효율적
# 4. 비용 절감: 불필요하게 긴 컨텍스트를 LLM에 전달하면 토큰 비용 증가

# chunk_size와 chunk_overlap 설정의 영향:
# - chunk_size가 너무 작으면: 의미 단위가 분할될 수 있음 (불완전한 문장)
# - chunk_size가 너무 크면: 검색 정확도 저하, LLM 처리 비용 증가
# - chunk_overlap가 적으면: 문맥 단절 가능성 증가
# - chunk_overlap가 크면: 중복 정보 증가, 효율성 감소

# 다양한 텍스트 분할기 종류:
# 1. CharacterTextSplitter: 고정 길이로 단순 분할 (의미 무시)
# 2. TokenTextSplitter: 토큰 수 기준 분할 (LLM 토큰 제한에 최적)
# 3. SentenceTransformersTokenTextSplitter: 임베딩 모델에 최적화된 분할
# 4. MarkdownTextSplitter: 마크다운 구조를 고려한 분할
# 5. PythonCodeTextSplitter: 파이썬 코드 구조 고려 분할

# 분할 시 고려사항:
# 1. 문서 유형: 법률 문서(조항), 논문(섹션), 코드(함수) 등에 따라 최적 분할 방식 다름
# 2. 언어 특성: 영어(공백 기준) vs 한국어(어절 기준) 분할 방식 차이
# 3. 구조 정보: 표, 목록, 수식 등 특수 요소 처리

# 분할 결과의 품질 검증 방법:
# 1. 청크 길이 분포 확인: 너무 짧거나 긴 청크 없는지
# 2. 의미 완전성 확인: 문장 중간에 끊기지 않았는지
# 3. 겹침 영역 검토: 중요한 정보가 손실되지 않았는지

# 실제 프로덕션 환경에서의 최적화:
# 1. 동적 chunk_size: 문서 유형별로 다른 크기 설정
# 2. 계층적 분할: 큰 단위(섹션) → 중간 단위(문단) → 작은 단위(문장)
# 3. 의미 기반 분할: NLP 모델을 사용한 의미적 경계 인식

# 분할 후 각 청크의 메타데이터:
# - 원본 소스 정보 유지: metadata['source'] = 'company_policy.txt'
# - 위치 정보 추가: metadata['chunk_index'] = 0, 1, 2, ...
# - 페이지 번호: 멀티페이지 문서인 경우

# 문제 상황 및 해결 방안:
# 1. 표나 그림 분할: 특수 처리가 필요한 요소는 별도 추출
# 2. 긴 단어나 URL: chunk_size보다 긴 단일 요소는 강제 분할 필요
# 3. 혼합 언어 문서: 언어별로 다른 분할 규칙 적용 필요

# 분할 단계의 성능 영향:
# 잘 분할된 문서는 검색기의 유사도 계산 정확도를 높이고,
# LLM에게 제공되는 컨텍스트의 관련성을 향상시킵니다.

# 다음 단계를 위한 준비:
# 분할된 각 청크는 독립적인 Document 객체가 되어,
# 다음 단계에서 각각 임베딩으로 변환되고 벡터 DB에 저장됩니다.

# 실험적 접근:
# 최적의 chunk_size와 chunk_overlap은 문서 유형과 사용 사례에 따라 다르므로,
# A/B 테스트를 통해 다양한 설정을 실험해보는 것이 좋습니다.

# 이 분할 단계가 RAG 시스템 성능에 미치는 핵심 영향:
# 1. 검색 정확도: 적절한 크기의 청크는 질문과의 의미적 유사성 계산에 유리
# 2. 답변 품질: 관련성 높은 청크만 제공받은 LLM은 더 정확한 답변 생성
# 3. 시스템 효율성: 불필요한 정보 전달 감소로 처리 속도 향상 및 비용 절감


 문서를 작은 조각(Chunk)으로 나눕니다...
   -> 총 5개의 조각(Chunk)으로 분할되었습니다.


### 임베딩 & 벡터 저장소 생성 (Indexing)

In [ ]:
# ==========================================
# 3. 임베딩 & 벡터 저장소 생성 (Indexing)
# ==========================================
# RAG 파이프라인의 세 번째 단계: 텍스트를 숫자 벡터로 변환하고 벡터 데이터베이스에 저장
# 이 단계는 문서 검색을 위한 "색인 생성" 과정입니다

print("\n 문서를 벡터로 변환하여 저장합니다...")

# OpenAIEmbeddings 객체 생성
# - model="text-embedding-3-small": OpenAI의 가성비 좋은 임베딩 모델
#   - 차원: 1536차원 (기본값, 필요시 더 작게 설정 가능)
#   - 특징: 빠르고 정확하며 비용 효율적
#   - 입력 텍스트 길이: 최대 8191 토큰
# - OpenAIEmbeddings 클래스는 자동으로 os.environ["OPENAI_API_KEY"]를 사용하여 인증

# Chroma.from_documents() 메서드를 사용하여:
# 1. 각 문서 청크를 임베딩 벡터로 변환 (OpenAI API 호출)
# 2. 변환된 벡터들을 Chroma 벡터 데이터베이스에 저장
# 3. 텍스트 청크와 해당 벡터를 매핑하여 저장
vectorstore = Chroma.from_documents(
    documents=splits,  # 분할된 문서 청크들 (Document 객체 리스트)
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")  # 임베딩 모델
)

print("   -> Chroma DB에 저장 완료!")

# 임베딩(Embedding)이란?
# - 텍스트를 고차원 공간의 숫자 벡터로 변환하는 과정
# - 의미적으로 유사한 텍스트는 벡터 공간에서 가까이 위치
# - 벡터 간 거리 계산(유사도 계산)으로 텍스트 간 관련성 측정

# OpenAI 임베딩 모델의 작동 원리:
# 1. 입력: 텍스트 문자열 (예: "안식월 1개월을 유급으로 제공한다.")
# 2. 처리: OpenAI 서버에서 딥러닝 모델이 텍스트를 분석
# 3. 출력: 1536개의 실수로 구성된 벡터 (예: [0.123, -0.456, ..., 0.789])
# 4. 특징: 문맥적 의미를 보존하는 dense vector 생성

# Chroma 벡터 데이터베이스의 역할:
# 1. 벡터 저장: 각 문서 청크의 임베딩 벡터를 효율적으로 저장
# 2. 유사도 검색: 쿼리와 유사한 벡터를 빠르게 찾기 위한 인덱싱 구조
# 3. 메타데이터 관리: 각 벡터에 해당하는 텍스트와 메타데이터 저장
# 4. 실시간 검색: 사용자 질문과 가장 유사한 문서를 밀리초 단위로 검색

# Chroma.from_documents()의 내부 처리 과정:
#
# 1. 임베딩 생성 단계:
#    - splits 리스트의 각 Document 객체에 대해:
#      a. page_content 텍스트 추출
#      b. OpenAI API 호출하여 임베딩 벡터 생성 (HTTP 요청)
#      c. 벡터와 텍스트 쌍을 임시 저장
#
# 2. 벡터 저장소 생성 단계:
#    - 메모리 내 Chroma 인스턴스 생성
#    - 벡터와 텍스트를 인덱싱 구조에 저장
#    - 기본적으로 메모리에 저장되지만 디스크 지속성 옵션 가능
#
# 3. 검색 최적화:
#    - HNSW(Hierarchical Navigable Small World) 같은 알고리즘으로 인덱스 구축
#    - 빠른 최근접 이웃 검색을 위한 데이터 구조 준비

# 벡터 데이터베이스 선택 기준:
# 1. Chroma: 경량, 오픈소스, 로컬 실행 용이 (이 예제에서 사용)
# 2. Pinecone: 관리형 클라우드 서비스, 대규모 데이터셋에 적합
# 3. Weaviate: 그래프 기능 포함, 다양한 모듈 지원
# 4. Qdrant: Rust 기반 고성능, 필터링 기능 강력
# 5. FAISS: Facebook 개발, 연구 목적에 적합

# 임베딩 모델 선택 기준:
# 1. OpenAI 임베딩: 정확도 높음, 비용 발생, API 의존성
# 2. Sentence-Transformers: 무료, 로컬 실행 가능, 다양한 언어 지원
# 3. Cohere 임베딩: 상업용 API, 다국어 지원 우수
# 4. Google의 Universal Sentence Encoder: 구글 클라우드 환경에 적합

# 벡터 차원(Dimension)의 중요성:
# - text-embedding-3-small: 1536차원 (기본)
# - 차원이 높을수록: 더 풍부한 의미 표현 가능, but 저장 공간과 계산 비용 증가
# - 차원이 낮을수록: 효율성 증가, but 의미 표현력 감소 가능성

# 임베딩 생성 비용과 시간:
# - 비용: 약 $0.02/1백만 토큰 (text-embedding-3-small 기준)
# - 시간: 네트워크 지연 + 모델 처리 시간 (일반적으로 청크당 100-500ms)
# - 배치 처리: 한 번에 여러 청크를 보내 비용과 시간 절감 가능

# 메타데이터 저장:
# Chroma는 각 벡터에 다음 메타데이터를 자동 저장:
# - source: 문서 원본 파일명
# - text: 청크의 텍스트 내용
# - 추가 사용자 정의 메타데이터 가능 (예: 문서 유형, 작성 날짜 등)

# 벡터 검색의 수학적 원리:
# 1. 코사인 유사도: 벡터 간 각도의 코사인 값으로 유사도 측정 (0~1)
#    - 1: 완전히 동일한 방향 (매우 유사)
#    - 0: 수직 방향 (무관계)
#    - -1: 정반대 방향 (반대 의미)
# 2. 유클리드 거리: 벡터 간 직선 거리 (거리가 가까울수록 유사)

# 실시간 업데이트와 확장성:
# - 새로운 문서 추가: vectorstore.add_documents()로 실시간 추가 가능
# - 문서 업데이트: 문서 ID로 특정 문서 삭제 후 새 버전 추가
# - 대규모 데이터셋: 수백만 개의 벡터까지 효율적으로 처리 가능

# 성능 최적화 팁:
# 1. 배치 임베딩: 여러 문서를 한 번의 API 호출로 임베딩 생성
# 2. 벡터 압축: PQ(Product Quantization)로 벡터 크기 축소
# 3. 인덱스 튜닝: HNSW 파라미터(M, ef_construction) 조정

# 오류 처리 및 모니터링:
# 1. API 제한: OpenAI의 분당/일일 요청 제한 고려
# 2. 네트워크 오류: 재시도 로직 구현 필요
# 3. 비용 모니터링: 임베딩 생성 토큰 수 추적

# 보안 고려사항:
# 1. 민감 정보: 개인정보가 포함된 문서는 임베딩 생성 전 마스킹 필요
# 2. API 키 보호: 임베딩 생성 중 API 키 노출 방지
# 3. 데이터 유출: 벡터 DB 접근 권한 제어

# 다음 단계(검색기 설정)를 위한 준비:
# vectorstore 객체는 검색기(Retriever)로 변환 가능
# 검색기는 사용자 질문을 임베딩으로 변환하고 벡터 DB에서 유사한 문서 검색

# 이 단계의 중요성:
# 임베딩 품질이 RAG 시스템의 검색 정확도를 결정합니다.
# 좋은 임베딩은 의미적 유사성을 잘 포착하여 관련성 높은 문서를 찾아줍니다.

# 시스템 아키텍처에서의 위치:
# 이 단계는 일반적으로 "오프라인 처리"로 분류됩니다.
# 문서가 변경되지 않는 한 주기적으로 또는 한 번만 실행하면 됩니다.

# 저장소 지속성 옵션:
# 기본적으로 메모리에 저장되지만, 디스크에 저장하려면:
# vectorstore = Chroma.from_documents(
#     documents=splits,
#     embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
#     persist_directory="./chroma_db"  # 디스크 저장 경로
# )
# vectorstore.persist()  # 명시적 저장


 문서를 벡터로 변환하여 저장합니다...
   -> Chroma DB에 저장 완료!


### 검색기(Retriever) 설정 & 생성 (Generation)

In [ ]:
# ==========================================
# 4. 검색기(Retriever) 설정 & 생성 (Generation)
# ==========================================
# RAG 파이프라인의 네 번째 단계: 검색기 설정과 생성 체인 구성
# 이 단계에서는 벡터 DB를 검색기로 변환하고, LLM과 함께 RAG 체인을 구성합니다

print("\n RAG 체인을 생성합니다...")

# 1. 검색기(Retriever) 설정
# vectorstore.as_retriever(): 벡터 저장소를 검색기 객체로 변환
# search_kwargs={"k": 3}: 가장 유사한 문서 3개를 검색하도록 설정
# - k=3: 상위 3개의 가장 관련성 높은 문서 조각을 검색
# - k 값은 실험을 통해 최적화 필요 (너무 작으면 정보 부족, 너무 크면 노이즈 증가)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. LLM 모델 설정
# ChatOpenAI: OpenAI의 채팅 모델을 사용하는 클래스
# - model="gpt-4o-mini": GPT-4o-mini 모델 사용 (가성비 좋은 최신 모델)
# - temperature=0: 창의성을 최소화하여 결정적인 답변 생성 (규정 답변에 적합)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 검색기(Retriever)의 역할과 작동 원리:
#
# Retriever는 사용자 질문을 받아 다음과 같은 과정을 거쳐 관련 문서를 검색합니다:
#
# 1. 질문 임베딩 생성: 사용자 질문을 벡터로 변환 (동일한 임베딩 모델 사용)
# 2. 유사도 검색: 벡터 DB에서 질문 벡터와 가장 유사한 k개의 문서 벡터 찾기
# 3. 문서 반환: 검색된 문서들의 텍스트 내용과 메타데이터 반환
#
# 예시:
# 질문: "입사 3년 되면 어떤 혜택이 있어?" → 임베딩 → 벡터 DB 검색 → 상위 3개 문서 반환

# 검색기 설정의 세부 옵션들:
# 1. search_type: "similarity"(유사도 기준), "mmr"(다양성 고려), "similarity_score_threshold"(점수 임계값)
# 2. search_kwargs:
#    - k: 검색할 문서 수
#    - score_threshold: 유사도 점수 기준치 설정 (이상인 문서만 반환)
#    - filter: 메타데이터 기반 필터링 (예: {"department": "HR"})

# 검색 알고리즘 종류:
# 1. 유사도 검색(Similarity Search): 코사인 유사도 기준으로 가장 가까운 벡터 검색
# 2. MMR(Maximal Marginal Relevance): 유사도와 다양성을 동시에 고려한 검색
# 3. 자체 정의 거리 함수: 유클리드 거리, 맨해튼 거리 등 다른 거리 척도 사용

# k 값(검색 문서 수) 설정의 중요성:
# - 너무 작은 k(예: 1): 질문에 대한 충분한 맥락 정보 부족
# - 너무 큰 k(예: 10): 관련 없는 정보 포함 가능성 증가, 토큰 낭비
# - 일반적으로 3-5개가 좋은 시작점이며, 실험을 통해 최적값 찾기

# LLM 모델 선택 기준:
# 1. GPT-4o-mini: 가성비 최적화, 빠른 응답, 128K 컨텍스트 윈도우
# 2. GPT-4: 더 높은 추론 능력, 비용 더 높음
# 3. GPT-3.5-turbo: 더 저렴하지만 성능 낮음
# 4. 기타 모델: Claude, Gemini 등 상황에 맞춰 선택

# temperature 파라미터의 의미:
# - 0: 확률이 높은 토큰만 선택 (일관된 답변)
# - 0.5: 적당한 창의성
# - 1.0: 높은 창의성, 다양한 답변 가능
# - 규정 챗봇은 정확성이 중요하므로 temperature=0 권장

# 검색기의 내부 메커니즘 상세:
#
# retriever.invoke("질문") 호출 시:
# 1. 질문 텍스트를 임베딩 모델에 전달하여 벡터화
# 2. 벡터 DB의 인덱스(HNSW 등)를 사용하여 k개의 가장 가까운 벡터 검색
# 3. 각 검색 결과는 Document 객체로 구성됨:
#    - page_content: 문서 텍스트
#    - metadata: 소스 정보, 유사도 점수 등
#    - 유사도 점수는 기본적으로 반환되지 않지만, 설정에 따라 포함 가능

# 검색 결과의 품질 평가 지표:
# 1. 재현율(Recall): 관련 있는 문서를 얼마나 잘 찾아내는지
# 2. 정밀도(Precision): 찾아낸 문서 중 실제 관련 있는 문서 비율
# 3. 평균 정밀도(MAP): 순위를 고려한 평가 지표

# 검색 성능 최적화 기법:
# 1. 하이브리드 검색: 벡터 검색 + 키워드 검색(BM25) 조합
# 2. 리랭킹(Reranking): 1차 검색 결과를 더 정교한 모델로 재정렬
# 3. 쿼리 확장: 원본 질문을 확장하거나 변형하여 검색 품질 향상
# 4. 필터링: 메타데이터 기반으로 검색 범위 제한

# LLM 모델의 컨텍스트 관리:
# - GPT-4o-mini: 128,000 토큰 제한
# - 검색된 문서 + 프롬프트 + 질문의 총 토큰 수가 제한을 초과하지 않도록 관리 필요
# - 토큰 수 계산: tiktoken 라이브러리로 정확한 토큰 수 확인 가능

# 검색 실패 시나리오 및 처리:
# 1. 관련 문서 없음: 검색 결과가 없거나 점수가 너무 낮은 경우
# 2. 문서 품질 문제: 문서가 너무 짧거나 정보가 부족한 경우
# 3. 쿼리 이해 실패: 질문이 모호하거나 복잡하여 검색 실패

# 검색 결과의 시각화와 디버깅:
# 1. 검색 점수 확인: 각 문서의 유사도 점수 출력하여 품질 확인
# 2. 검색된 문서 내용 확인: 어떤 내용이 검색되었는지 직접 확인
# 3. 임베딩 시각화: t-SNE, PCA로 고차원 벡터를 2D로 시각화

# 검색기와 LLM의 연동 준비:
# 이제 retriever는 질문 → 관련 문서를 검색하는 기능을 갖추었고,
# llm은 문서를 기반으로 답변을 생성하는 기능을 갖추었습니다.
# 다음 단계에서는 이 둘을 연결하는 프롬프트 템플릿과 체인을 구성합니다.

# 실제 프로덕션 환경에서의 고려사항:
# 1. 검색 지연 시간: 사용자 대기 시간을 고려한 k 값과 알고리즘 선택
# 2. 확장성: 대규모 문서 집합에서의 검색 성능 보장
# 3. 비용 관리: 임베딩 API 호출 비용과 LLM 토큰 비용 최적화

# 보안 및 접근 제어:
# 1. 사용자별 문서 접근 권한: 메타데이터 필터링으로 구현 가능
# 2. 감사 로그: 어떤 질문으로 어떤 문서가 검색되었는지 기록
# 3. 민감 정보 검색 제한: 특정 문서는 검색에서 제외

# 다음 단계에서 수행할 작업:
# 1. 프롬프트 템플릿 작성: 검색된 문서(context)와 질문(question)을 포함할 프롬프트
# 2. 체인 구성: retriever → prompt → llm → parser 연결
# 3. 테스트: 실제 질문으로 시스템 테스트

# 이 단계의 중요성:
# 검색기의 성능이 RAG 시스템 전체의 성능을 결정합니다.
# 좋은 검색기는 질문의 의도를 정확히 이해하고 가장 관련성 높은 문서를 찾아냅니다.


 RAG 체인을 생성합니다...


In [ ]:
# 프롬프트 템플릿 작성
# 이 부분은 검색된 문서와 사용자 질문을 결합하여 LLM에 전달할 최종 프롬프트를 설계합니다
# 프롬프트 템플릿은 RAG 시스템의 "증강(Augmented)" 단계에서 핵심 역할을 합니다

# 프롬프트 템플릿 문자열 정의
# {context}: 검색기(Retriever)가 가져온 문서 내용이 들어갈 자리표시자
# {question}: 사용자의 원본 질문이 들어갈 자리표시자
template = """
당신은 회사의 인사 담당 AI입니다.
아래 [규정]을 참고하여 직원의 질문에 친절하게 답해주세요.
규정에 없는 내용은 "규정에 나와있지 않습니다"라고 답하세요.

[규정]
{context}

[질문]: {question}
"""

# 당신은 회사의 인사 담당 AI입니다.  # 1. 역할 정의: LLM의 정체성과 응답 스타일 설정
# 아래 [규정]을 참고하여 직원의 질문에 친절하게 답해주세요.  # 2. 작업 지시: 컨텍스트 사용 방법 명시
# 규정에 없는 내용은 "규정에 나와있지 않습니다"라고 답하세요.  # 3. 안전 장치: 환각(hallucination) 방지 지시
# [규정]  # 4. 컨텍스트 영역 표시: LLM이 참고할 정보의 경계를 명확히 함
# {context}  # 5. 실제 검색된 문서 내용이 삽입될 위치 (동적 데이터)
# [질문]: {question}  # 6. 사용자 질문이 삽입될 위치

# PromptTemplate.from_template() 메서드로 템플릿 객체 생성
# 이 객체는 템플릿 문자열을 분석하고, {context}, {question} 같은 변수들을 인식합니다
# 생성된 객체는 나중에 실제 값으로 변수를 치환할 수 있는 format() 메서드를 제공합니다
prompt = PromptTemplate.from_template(template)

# 프롬프트 템플릿 설계의 심리학적 원리:
# 1. 역할 부여 효과: "인사 담당 AI"라는 역할은 LLM이 공식적이고 전문적인 답변을 생성하도록 유도
# 2. 친절성 강조: "친절하게 답해주세요"는 LLM의 응답 톤을 부드럽고 도움되는 스타일로 조정
# 3. 정보 제한: "규정에 없는 내용은..."은 LLM의 자발적 지식 활용을 제한하여 정확성 보장

# 프롬프트 엔지니어링의 중요성:
# 잘 설계된 프롬프트는 LLM이 컨텍스트를 효과적으로 활용하고, 일관된 형식으로 답변하도록 합니다.
# 프롬프트의 각 부분은 특정 목적을 가지고 있으며, 누락되면 원하지 않는 응답이 발생할 수 있습니다.

# 템플릿 변수의 동적 채우기 과정:
#
# [실행 전 - 템플릿 상태]
# template 문자열에 {context}와 {question}이 자리표시자로 존재
#
# [실행 시 - 변수 치환]
# 1. retriever가 검색한 문서들이 하나의 문자열로 결합되어 context 변수값이 됨
# 2. 사용자의 원본 질문이 question 변수값이 됨
# 3. prompt.format(context=검색문서, question=사용자질문) 호출
# 4. {context}가 실제 검색 문서로, {question}이 실제 질문으로 대체됨
#
# [실행 후 - 완성된 프롬프트 예시]
# """
# 당신은 회사의 인사 담당 AI입니다.
# 아래 [규정]을 참고하여 직원의 질문에 친절하게 답해주세요.
# 규정에 없는 내용은 "규정에 나와있지 않습니다"라고 답하세요.
#
# [규정]
# 2. 3년 이상 근속 시 안식월 1개월을 유급으로 제공한다.
# 1. 도서 구입비는 월 10만원 한도 내에서 무제한 지원한다.
#
# [질문]: 입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?
# """

# 컨텍스트(context) 변수의 구조와 형식:
# - retriever는 3개의 Document 객체를 반환 (k=3 설정)
# - 각 Document의 page_content가 추출되어 하나의 문자열로 결합
# - 결합 방식은 기본적으로 "\n\n"으로 구분됨
# - 실제로는 더 정교한 컨텍스트 구성 전략이 필요할 수 있음 (예: 관련성 순 정렬)

# 프롬프트 템플릿의 최적화 기술:
# 1. Few-shot 프롬프팅: 예시 질문과 답변을 포함시켜 LLM의 응답 형식 안내
# 2. 체인 오브 씽킹(Chain-of-Thought): 단계적 추론을 유도하는 지시문 추가
# 3. 출력 형식 지정: JSON, 마크다운 등 특정 형식으로 답변하도록 요청
# 4. 조건부 지시: 질문 유형에 따라 다른 지시문 적용

# 실제 프로덕션 환경에서의 고려사항:
# 1. 다국어 지원: 사용자 언어에 맞는 프롬프트 템플릿 필요
# 2. 개인화: 사용자 역할(임직원, 관리자, 신입사원)에 따른 맞춤형 응답
# 3. 법적 완화: 법률적 조언이 아닌 정보 제공임을 명시

# 프롬프트 주입(Prompt Injection) 방어:
# 1. 입력 검증: 사용자 질문에 시스템 지시를 변경하는 내용이 있는지 검사
# 2. 컨텍스트 분리: 사용자 입력과 시스템 지시를 명확히 구분
# 3. 권한 제한: 민감한 작업 수행을 위한 특별 지시문 방지

# 프롬프트 길이와 토큰 제한 관리:
# - 검색된 문서가 많을수록 컨텍스트가 길어져 토큰 수 증가
# - GPT-4o-mini는 128K 토큰 제한이 있지만, 실제로는 비용과 속도를 고려하여 제한 필요
# - 컨텍스트 압축: 긴 문서를 요약하거나 가장 관련성 높은 부분만 추출

# 컨텍스트 구성 전략:
# 1. 관련성 정렬: 유사도 점수가 높은 순으로 문서 정렬
# 2. 중복 제거: 동일하거나 매우 유사한 내용의 문서 제거
# 3. 메타데이터 활용: 문서 유형, 신뢰도에 따른 가중치 적용
# 4. 청크 윈도우: 검색된 청크의 앞뒤 추가 문맥 포함

# 프롬프트 템플릿의 테스트와 평가:
# 1. 다양한 질문 유형으로 테스트: 사실 질문, 해석 질문, 복합 질문 등
# 2. 모호성 처리: 불명확한 질문에 대한 확인 질문 생성 능력 평가
# 3. 일관성 검증: 동일 질문에 대한 반복 응답의 일관성 확인

# 프롬프트 버전 관리:
# - 프롬프트 변경은 응답 품질에 큰 영향을 미침
# - 변경 사항을 추적하고, A/B 테스트를 통해 효과 검증 필요
# - Git 등 버전 관리 시스템에 프롬프트 템플릿 저장 권장

# 확장 가능한 프롬프트 아키텍처:
# 1. 조건부 분기: 질문 카테고리별 다른 템플릿 사용
# 2. 동적 지시문: 검색 결과의 특성에 따라 지시문 조정
# 3. 멀티스텝 프롬프트: 여러 단계의 프롬프트를 순차적 적용

# 프롬프트 디버깅 도구:
# 1. 프롬프트 로깅: 실제로 LLM에 전송되는 프롬프트 기록
# 2. 변수 값 확인: context와 question 변수의 실제 값 확인
# 3. 응답 분석: 특정 프롬프트 구성이 응답에 미치는 영향 분석

# 이 프롬프트 템플릿이 RAG 체인에서 수행하는 핵심 기능:
# "검색된 지식(Retrieval) + 사용자 질문 → 구조화된 프롬프트로 증강 → LLM 전달"
#
# 이는 오픈북 시험에서 학생(LLM)에게 문제지(질문)와 교과서(컨텍스트)를 함께 주는 것과 같습니다.

# 다음 단계에서의 활용:
# 이 prompt 객체는 LCEL(체인 표현 언어) 파이프라인에서 사용되며,
# retriever의 출력과 사용자 질문을 받아 최종 프롬프트를 생성합니다.

# 실패 시나리오 및 대응:
# 1. 컨텍스트가 비어있는 경우: 검색 결과가 없을 때의 처리 로직 필요
# 2. 상충되는 정보: 서로 다른 문서가 모순되는 정보를 포함할 경우
# 3. 낮은 신뢰도: 검색 점수가 낮은 문서들만 있는 경우

# 이 프롬프트 템플릿의 진화 가능성:
# 1. 도메인 특화: 인사 규정 외 다른 규정(보안, 재무 등)에 대한 템플릿 분화
# 2. 대화형 확장: 이전 대화 맥락을 포함하여 연속된 대화 지원
# 3. 시각적 요소: 표, 차트 등 구조화된 정보 처리 지시 추가

# 프롬프트 템플릿의 성공 기준:
# 1. 명확성: LLM이 지시사항을 명확히 이해할 수 있어야 함
# 2. 효과성: 원하는 형식과 내용의 응답을 이끌어내야 함
# 3. 강건성: 다양한 입력에 대해 일관된 품질 유지
# 4. 효율성: 불필요한 지시문이나 반복을 최소화

# 이 단계의 중요성:
# 프롬프트 템플릿은 검색과 생성 단계를 연결하는 다리 역할을 합니다.
# 잘 설계된 템플릿은 LLM이 검색된 정보를 최대한 활용하도록 안내하고,
# 사용자에게 유용하고 정확한 답변을 제공하는 데 결정적 역할을 합니다.

In [ ]:
# 체인 연결 (Retrieval -> Prompt -> LLM -> Parser): LangChain Expression Language(LCEL)
# 이 코드는 RAG 시스템의 전체 파이프라인을 하나의 실행 가능한 체인으로 구성합니다
# LCEL은 함수형 프로그래밍 스타일로 컴포넌트들을 연결하는 직관적인 방법을 제공합니다

# RunnablePassthrough: 질문을 그대로 전달하고 싶은 경우 사용
# RunnableLambda: 질문을 가공하고 싶을 경우 사용 (예: 전처리, 확장 등)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}  # 1. 입력 데이터 구성
    | prompt  # 2. 프롬프트 템플릿 적용
    | llm  # 3. LLM 모델 호출
    | StrOutputParser()  # 4. 출력 문자열로 파싱
)





# LCEL 체인의 구성 요소별 상세 설명:
#
# 1. 입력 데이터 구성 단계: {"context": retriever, "question": RunnablePassthrough()}
#    - 이 부분은 딕셔너리를 생성하여 두 개의 입력 경로를 정의합니다:
#      a. "context": retriever 객체 - 질문을 받아 관련 문서를 검색하는 함수 역할
#      b. "question": RunnablePassthrough() 객체 - 입력값(질문)을 변경 없이 전달
#
#    - 실제 실행 시 이 딕셔너리는 다음과 같이 처리됩니다:
#      입력(질문 문자열) →
#        retriever(질문) 실행 → 문서 검색 결과
#        RunnablePassthrough()(질문) 실행 → 동일한 질문 문자열
#      → {"context": [문서1, 문서2, ...], "question": "원본질문"}
#
# 2. 프롬프트 적용 단계: | prompt
#    - 파이프(|) 연산자는 왼쪽의 출력을 오른쪽의 입력으로 전달합니다
#    - prompt 객체는 {"context": ..., "question": ...} 딕셔너리를 받아:
#      a. context 값을 문서들의 텍스트로 변환하여 하나의 문자열로 결합
#      b. question 값을 그대로 가져옴
#      c. 템플릿의 {context}와 {question} 자리표시자를 실제 값으로 대체
#      d. 완성된 프롬프트 문자열 생성
#
# 3. LLM 호출 단계: | llm
#    - 프롬프트 문자열을 ChatOpenAI 객체에 전달
#    - OpenAI API를 호출하여 LLM 모델 실행
#    - ChatMessage 객체 형태로 응답 수신
#
# 4. 출력 파싱 단계: | StrOutputParser()
#    - ChatMessage 객체에서 실제 텍스트 내용 추출
#    - 최종적으로 순수 문자열로 변환하여 반환

# LCEL의 내부 동작 메커니즘:
#
# [체인 생성 시점]
# 1. 각 컴포넌트는 Runnable 인터페이스를 구현
# 2. 파이프 연산자는 RunnableSequence 객체를 생성
# 3. 각 단계의 입력/출력 타입이 호환되는지 확인 (런타임 전)
#
# [체인 실행 시점 (invoke() 호출 시)]
# 1. rag_chain.invoke("질문") 호출
# 2. 입력 "질문"이 첫 번째 컴포넌트(딕셔너리)로 전달
# 3. 딕셔너리의 각 값이 병렬 또는 순차적으로 실행:
#    - retriever.invoke("질문") → 문서 리스트 반환
#    - RunnablePassthrough().invoke("질문") → "질문" 그대로 반환
# 4. 결과 딕셔너리가 prompt.invoke()로 전달
# 5. prompt.invoke()가 완성된 프롬프트 문자열 반환
# 6. llm.invoke()가 프롬프트를 처리하여 ChatMessage 반환
# 7. StrOutputParser().invoke()가 ChatMessage를 문자열로 변환
# 8. 최종 문자열 반환

# RunnablePassthrough의 정확한 역할:
# - 단순히 입력값을 변경 없이 전달하는 "패스스루" 컴포넌트
# - 질문을 가공하지 않고 그대로 다음 단계에 전달하고 싶을 때 사용
# - 예시: 입력 "안녕하세요" → 출력 "안녕하세요" (변화 없음)

# RunnableLambda와의 비교:
# - RunnablePassthrough: 입력 그대로 전달
# - RunnableLambda: 사용자 정의 함수를 적용하여 입력 변환
#   예: RunnableLambda(lambda x: x.upper()) → 입력을 대문자로 변환

# 딕셔너리 구성의 의미:
# {"context": retriever, "question": RunnablePassthrough()}
# - 이 구조는 prompt 템플릿이 기대하는 입력 형식({context}, {question})에 맞춤
# - retriever는 함수처럼 동작: 입력(질문) → 출력(문서들)
# - RunnablePassthrough()는 항등 함수처럼 동작: 입력 → 동일한 출력

# 병렬 실행 vs 순차 실행:
# 딕셔너리의 값들은 이론적으로 병렬 실행이 가능하지만,
# 실제로는 LangChain의 구현에 따라 순차 실행될 수 있습니다.
# 중요한 것은 retriever와 RunnablePassthrough()가 동일한 입력(질문)을 받는다는 점입니다.

# 체인의 확장성과 유연성:
# 1. 추가 전처리 단계 삽입:
#    rag_chain = (
#        {"context": retriever, "question": RunnablePassthrough()}
#        | {"context": lambda x: x["context"], "question": lambda x: x["question"].strip()}
#        | prompt | llm | StrOutputParser()
#    )
#
# 2. 여러 검색기 결합:
#    {"context": retriever1 | retriever2, "question": RunnablePassthrough()}
#
# 3. 조건부 분기:
#    from langchain_core.runnables import RunnableBranch
#    branch = RunnableBranch(...)

# 오류 처리와 복원력:
# 1. 각 단계에서 예외 발생 시 전체 체인 실패
# 2. try-catch를 통한 오류 처리나 fallback 체인 구성 가능
# 3. 타임아웃 설정으로 무한 대기 방지

# 성능 최적화 기회:
# 1. 배치 처리: 여러 질문을 한 번에 처리
# 2. 캐싱: 동일 질문에 대한 검색 결과 캐시
# 3. 비동기 처리: await를 사용한 비동기 호출

# 디버깅과 모니터링:
# 1. 중간 값 확인: 각 단계의 입력/출력을 로깅
# 2. 실행 시간 측정: 각 컴포넌트의 처리 시간 기록
# 3. 토큰 사용량 추적: LLM 호출 비용 모니터링

# 실제 실행 흐름 예시:
#
# 입력: "입사 3년 되면 어떤 혜택이 있어?"
#
# 단계 1: {"context": retriever, "question": RunnablePassthrough()}
#   - retriever("입사 3년 되면 어떤 혜택이 있어?") → [문서A, 문서B, 문서C]
#   - RunnablePassthrough()("입사 3년 되면 어떤 혜택이 있어?") → "입사 3년 되면 어떤 혜택이 있어?"
#   → 출력: {"context": [문서A, 문서B, 문서C], "question": "입사 3년 되면 어떤 혜택이 있어?"}
#
# 단계 2: prompt
#   - prompt.format(context="문서A내용\n문서B내용\n문서C내용", question="입사 3년 되면 어떤 혜택이 있어?")
#   → 출력: 완성된 프롬프트 문자열
#
# 단계 3: llm
#   - llm.invoke(완성된_프롬프트)
#   → 출력: ChatMessage(content="3년 이상 근속 시 안식월 1개월을 유급으로 제공합니다.")
#
# 단계 4: StrOutputParser()
#   - StrOutputParser().invoke(ChatMessage(...))
#   → 최종 출력: "3년 이상 근속 시 안식월 1개월을 유급으로 제공합니다."

# 체인의 재사용성:
# rag_chain 객체는 일단 생성되면:
# 1. 여러 번 재사용 가능 (동일한 설정으로 여러 질문 처리)
# 2. 저장하고 불러오기 가능 (pickle 또는 LangChain의 저장 기능)
# 3. 다른 체인과 결합 가능 (더 큰 워크플로우의 일부로 사용)

# 고급 LCEL 기능:
# 1. .map()을 통한 병렬 처리
# 2. .with_fallbacks()를 통한 폴백 체인
# 3. .with_config()를 통한 실행 설정
# 4. .stream()을 통한 스트리밍 출력

# 보안 고려사항:
# 1. 입력 검증: 사용자 질문에 악성 코드나 프롬프트 주입 시도 검사
# 2. 출력 검증: LLM 응답에 민감 정보나 부적절한 내용 포함 여부 검사
# 3. 접근 제어: 특정 사용자만 특정 문서에 접근 가능하도록 제한

# 이 체인이 실제 RAG 시스템에서의 위치:
# 이는 RAG의 "생성(Generation)" 단계를 포함한 전체 파이프라인입니다.
# 앞서의 문서 로딩, 분할, 임베딩, 인덱싱은 오프라인 전처리 단계였으며,
# 이 체인은 실시간 질의응답을 처리하는 온라인 서비스 부분입니다.

# 확장 가능한 아키텍처 패턴:
# 1. 마이크로서비스: 체인을 독립적인 서비스로 배포
# 2. 이벤트 기반: 메시지 큐를 통해 질문 수신 및 응답 발행
# 3. API 게이트웨이: REST 또는 GraphQL API로 체인 노출

# 테스트와 검증:
# 1. 단위 테스트: 각 컴포넌트별 독립 테스트
# 2. 통합 테스트: 전체 체인 종단간 테스트
# 3. 성능 테스트: 부하 테스트와 지연 시간 측정
# 4. A/B 테스트: 다른 설정의 체인 비교 평가

# 이 체인의 최종 목표:
# 사용자의 자연어 질문을 받아:
# 1. 벡터 DB에서 관련 문서 검색 (Retrieval)
# 2. 검색된 문서를 프롬프트에 포함 (Augmented)
# 3. LLM이 문서를 참고하여 답변 생성 (Generation)
# 4. 사용자에게 자연어 응답 제공

# 다음 단계에서는 이 체인을 실제로 실행하고 결과를 확인하게 됩니다.
# rag_chain.invoke("질문") 형태로 호출하여 최종 답변을 얻을 수 있습니다.

# 참고: 실제 프로덕션 환경에서는 이 체인에 추가적인 레이어가 필요할 수 있습니다:
# - 인증/인가: 사용자 식별과 권한 확인
# - 속도 제한: API 남용 방지
# - 로깅: 감사 추적과 분석
# - 모니터링: 시스템 건강 상태 추적
# - 경고: 이상 상황 감지 및 알림

# 이 체인 구성의 아름다움:
# 선언적 문법으로 복잡한 RAG 파이프라인을 간결하게 표현할 수 있으며,
# 각 컴포넌트를 독립적으로 교체하거나 확장하기 쉽습니다.

# 마지막으로, 이 체인은 RAG의 핵심 아이디어를 구현합니다:
# "외부 지식을 검색하여 LLM의 답변을 증강시킨다"

### 실행 (Test)

In [ ]:
# RAG 파이프라인의 마지막 단계: 실제 질문으로 시스템을 테스트하고 결과를 확인합니다
# 이 단계에서는 구성된 RAG 체인을 실제로 실행하여 사용자 질문에 대한 답변을 생성합니다

# 테스트용 질문 정의
# 이 질문은 두 부분으로 구성되어 RAG 시스템의 다양한 기능을 테스트합니다:
# 1. "입사한 지 3년 되면 어떤 혜택이 있어?" → 규정 조회 및 특정 조건에 따른 정보 검색
# 2. "그리고 책 사는 건 얼마나 지원해줘?" → 복합 질문 처리 및 여러 규정 조항 참조
question = "입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?"

# 질문 출력 (사용자에게 어떤 질문을 하는지 보여줌)
print(f"\n 질문: {question}")
print("AI 답변 생성 중...\n")

# RAG 체인 실행
# qa_chain.invoke(question): 구성된 체인에 질문을 입력하여 실행
# invoke 대신 stream을 쓰면 실시간으로 생성하는 과정이 보임
# 이 호출은 RAG 시스템 전체 파이프라인을 동작시키며, 내부적으로 다음과 같은 과정을 거칩니다:
# 1. 질문 임베딩 생성 및 유사도 검색
# 2. 검색된 문서를 컨텍스트로 LLM에 전달
# 3. LLM이 컨텍스트 기반 답변 생성
result = qa_chain.invoke(question)

# 답변 추출 (qa_chain은 {"answer": 답변, "source_documents": 문서} 형식으로 반환)
response = result["answer"]

# 결과 출력
# 구분선으로 시각적 구분을 주고, 생성된 답변을 출력합니다
print("="*50)
print("🤖 답변:")
print(response)
print("="*50)

# 참고 문서 정보 출력
sources = result["source_documents"]
print(f"\n📚 참고한 문서: {len(sources)}개")
for i, doc in enumerate(sources, 1):
    # 문서 메타데이터에서 질문 정보 추출
    q = doc.metadata.get("question", "(질문 정보 없음)")
    print(f"  {i}. {q}")
print("="*50)




# qa_chain.invoke(question)의 내부 실행 과정 상세 분석:

# [단계 0: 질문 입력]
# 입력값: "입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?"
# 실제 FAQ 데이터는 HR 관련이므로, 이 질문은 FAQ에 없을 가능성이 높음

# [단계 1: 병렬 실행 시작 - qa_chain 구조]
# qa_chain은 RunnableParallel로 구성:
#   - answer: answer_chain 실행 (답변 생성)
#   - source_documents: retriever 실행 (문서 검색)
# 두 실행이 병렬로 진행됨

# [단계 2: retriever 실행 (문서 검색)]
# 2.1 질문 임베딩 생성:
#     - "입사한 지 3년 되면..." → OpenAI 임베딩 API 호출
#     - 1536차원 벡터로 변환
#
# 2.2 벡터 유사도 검색:
#     - 질문 벡터와 FAQ 문서 벡터들(10개) 간 코사인 유사도 계산
#     - FAQ에는 "입사 3년" 관련 문서가 없으므로, 유사한 주제 검색
#     - 예상 검색 결과 (유사도 순):
#       1. "사내 교육 프로그램이 있나요?" (교육, 복지 관련성)
#       2. "복지 포인트는 어떻게 사용하나요?" (복지 관련성)
#       3. "건강검진은 언제 받나요?" (복지, 혜택 관련성)
#
# 2.3 상위 3개 문서 반환:
#     - [Document1, Document2, Document3]

# [단계 3: answer_chain 실행 (답변 생성)]
# 3.1 RunnableParallel 실행:
#     - context: retriever(질문) → 검색된 3개 문서
#     - question: RunnablePassthrough() → 원본 질문 그대로
#
# 3.2 프롬프트 구성 (rag_prompt):
#     입력: {"context": [문서1, 문서2, 문서3], "question": "입사한 지 3년 되면..."}
#
#     프롬프트 완성 예시:
#     """
#     당신은 회사의 HR 챗봇입니다.
#     아래 제공된 FAQ 정보를 바탕으로 직원의 질문에 정확하고 친절하게 답변해주세요.
#
#     FAQ 정보:
#     질문: 사내 교육 프로그램이 있나요?
#     답변: 분기별로 직무 교육과 온라인 교육 플랫폼을 제공합니다...
#
#     질문: 복지 포인트는 어떻게 사용하나요?
#     답변: 매년 1월 1일 기준으로 연간 150만원의 복지 포인트가 지급됩니다...
#
#     질문: 건강검진은 언제 받나요?
#     답변: 연간 건강검진은 입사 기념월에 받을 수 있습니다...
#
#     질문: 입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?
#
#     답변 가이드:
#     1. FAQ에 정확한 답변이 있으면 그대로 전달하세요
#     2. FAQ에 없는 내용이면 "정확한 정보는 인사팀(내선 1234)으로 문의해주세요"라고 안내하세요
#     3. 친절하고 전문적인 톤을 유지하세요
#
#     답변:
#     """
#
# 3.3 LLM 호출:
#     - 모델: gpt-4o-mini
#     - temperature: 0 (결정적 응답)
#     - LLM 분석:
#       * FAQ에 "입사 3년" 관련 정보 없음
#       * FAQ에 "책 구입" 관련 정보 없음
#       * FAQ에는 교육, 복지 포인트, 건강검진 정보만 있음
#     - 답변 생성: FAQ에 없는 내용 → 안내 메시지 생성
#
# 3.4 StrOutputParser 실행:
#     - LLM 응답(ChatMessage) → 문자열 변환

# [단계 4: 결과 병합]
# answer_chain 출력: "정확한 정보는 인사팀(내선 1234)으로 문의해주세요"
# retriever 출력: [문서1, 문서2, 문서3]
# 병합 결과: {"answer": "정확한 정보는...", "source_documents": [문서1, 문서2, 문서3]}

# [단계 5: 출력 분석]
# 예상 출력:
# ==================================================
# 🤖 답변:
# FAQ에 입사 3년차 혜택과 도서 구입 지원에 대한 구체적인 정보가 없습니다.
# 정확한 정보는 인사팀(내선 1234)으로 문의해주세요.
# ==================================================
#
# 📚 참고한 문서: 3개
#   1. 사내 교육 프로그램이 있나요?
#   2. 복지 포인트는 어떻게 사용하나요?
#   3. 건강검진은 언제 받나요?
# ==================================================

# RAG 시스템의 핵심 테스트 포인트:
# 1. 검색 정확도: FAQ에 없는 질문에 대해 가장 유사한 문서들을 찾았는가?
# 2. 컨텍스트 이해: LLM이 FAQ에 해당 정보가 없음을 인지했는가?
# 3. 환각 방지: 없는 정보를 지어내지 않고 안내 메시지를 생성했는가?
# 4. 형식 준수: 친절한 톤과 지정된 안내 메시지를 사용했는가?
# 5. 복합 질문 처리: 두 가지 질문을 모두 인지하고 처리했는가?

# 테스트 질문의 교육적 목적:
# 1. FAQ 범위 밖 질문 처리: 시스템이 한계를 정직하게 인정하는지 확인
# 2. 사용자 경험: 사용자를 적절한 채널(인사팀)로 안내하는지 확인
# 3. 시스템 신뢰성: 잘못된 정보를 생성하지 않는지 확인

# 실제 서비스 시나리오에서의 대응:
# 1. 즉시 응답: FAQ에 있는 정보는 즉시 답변
# 2. 적절한 안내: FAQ에 없는 정보는 담당 부서 안내
# 3. 피드백 수집: 자주 묻는 새로운 질문은 FAQ에 추가 검토

# 시스템 개선을 위한 인사이트:
# 1. FAQ 확장 필요성: 자주 묻는 새로운 질문 식별
# 2. 검색 품질: 유사도 임계값 조정 필요 여부 확인
# 3. 프롬프트 최적화: 안내 메시지의 명확성 검토

# 평가 지표:
# 1. 정확성: 잘못된 정보 제공 여부
# 2. 유용성: 사용자에게 도움이 되는 응답인지
# 3. 전문성: 전문적이고 신뢰할 수 있는 톤 유지
# 4. 응답 시간: 빠른 응답 제공 여부

# 확장 테스트 시나리오:
# 1. FAQ에 있는 질문 테스트: "연차 신청 방법"
# 2. 부분적으로 있는 질문 테스트: "교육 지원" (있음) + "주식 교육" (없음)
# 3. 모호한 질문 테스트: "혜택 알려줘" (너무 일반적)
# 4. 잘못된 질문 테스트: "우주여행 지원해줘?" (전혀 관련 없음)

# 실제 출력의 다양성:
# LLM이 생성하는 정확한 문구는 약간 다를 수 있지만, 핵심 메시지는 동일해야 함:
# - FAQ에 해당 정보 없음
# - 인사팀으로 문의 안내
# - 친절한 톤 유지

# 디버깅 정보 수집:
# 1. 검색된 문서 확인: 어떤 문서가 검색되었는지
# 2. 유사도 점수 확인: 검색된 문서들의 유사도 점수가 얼마나 높은지
# 3. 프롬프트 확인: LLM에 실제로 전송된 프롬프트 내용
# 4. LLM 응답 원본: 파싱 전 원본 응답 확인

# 성능 측정:
# 1. 전체 응답 시간: 질문 입력부터 응답 출력까지
# 2. 검색 시간: 벡터 검색에 걸린 시간
# 3. LLM 호출 시간: API 응답 대기 시간
# 4. 토큰 사용량: 입력/출력 토큰 수

# 실제 서비스 환경에서의 고려사항:
# 1. 사용자 경험: 안내 메시지가 너무 자주 나오지 않도록 FAQ 확장
# 2. 대체 채널: 인사팀 연락처 외에 자동 이메일 전송 등의 기능 고려
# 3. 로깅: 자주 묻는 새로운 질문을 자동으로 식별하여 FAQ 후보로 등록

# 반복적 개선 사이클:
# 1. 테스트 실행 → 2. 결과 분석 → 3. FAQ 데이터 확장 →
# 4. 시스템 재학습 → 5. 재테스트

# 출력 결과의 활용:
# 1. 최종 사용자에게 직접 표시
# 2. 사용자 피드백 수집 기반
# 3. 시스템 품질 모니터링
# 4. FAQ 지속적 개선을 위한 데이터 수집

# 이 테스트의 중요성:
# 이 테스트는 RAG 시스템이 실제 운영 환경에서 마주칠 수 있는 다양한 상황을 시뮬레이션합니다.
# 특히 FAQ 범위를 벗어난 질문에 대해 시스템이 어떻게 대응하는지 확인함으로써,
# 사용자에게 신뢰할 수 있는 서비스를 제공할 수 있는지 검증합니다.

# 실제 서비스에서는 이러한 테스트를 정기적으로 수행하여 시스템의 성능을 모니터링하고
# 지속적으로 개선해 나가야 합니다.


 질문: 입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?
AI 답변 생성 중...

입사한 지 3년이 지나면 안식월 1개월을 유급으로 제공받게 됩니다. 또한, 도서 구입비는 월 10만원 한도 내에서 무제한 지원됩니다.


In [ ]:
question = "입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?"

print(f"\n 질문: {question}")
print("AI 답변 생성 중...\n")

# stream() 메서드를 사용한 실시간 출력
# invoke()와 달리 stream()은 청크(chunk) 단위로 실시간 결과를 반환
# end="": 줄바꿈 없이 이어서 출력
# flush=True: 버퍼를 즉시 비워서 실시간 출력 보장
for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)

print("\n" + "="*30)
print(response)
print("="*30)




# rag_chain.stream(question)의 내부 실행 과정 및 실시간 출력 메커니즘:

# [stream() vs invoke() 차이점]
# invoke(): 전체 결과를 한 번에 반환 (동기적 실행)
# stream(): 결과를 작은 조각(chunk)으로 나누어 실시간 반환 (점진적 실행)

# [stream()의 실행 흐름]

# [단계 0: stream() 메서드 호출]
# rag_chain.stream(question) 호출 시:
# 1. 질문 처리 시작
# 2. 제너레이터(generator) 객체 반환
# 3. 실시간으로 결과 청크 생성 시작

# [단계 1: retriever 실행 (첫 번째 청크 생성 전)]
# 1.1 질문 임베딩 생성 (즉시 실행)
# 1.2 벡터 DB 검색 (즉시 실행)
# 1.3 검색 결과: [문서1, 문서2, 문서3]
# ※ retriever 실행은 stream() 시작 전에 완료됨 (실시간 아님)

# [단계 2: LLM 스트리밍 시작 (실시간 청크 생성)]
# 2.1 프롬프트 구성 및 LLM 호출 준비
# 2.2 OpenAI의 스트리밍 API 호출:
#     - stream=True 파라미터로 호출
#     - LLM이 토큰 단위로 점진적 응답 생성
#     - 각 토큰이 생성될 때마다 서버에서 클라이언트로 전송

# [단계 3: 실시간 청크 출력 과정]
# for chunk in rag_chain.stream(question): 루프 실행

# [청크 1]: 첫 번째 문장의 일부
# 출력: "FAQ에 입사 3년차 혜택과"
# 실제 동작: LLM이 첫 번째 토큰부터 순차적으로 생성

# [청크 2]: 첫 번째 문장의 나머지
# 출력: " 도서 구입 지원에 대한"
# 시간차: 약 100-300ms 후 다음 청크 도착

# [청크 3]: 두 번째 문장의 시작
# 출력: " 구체적인 정보가 없습니다."
# 프린터: print(chunk, end="", flush=True)로 이어서 출력

# [청크 4]: 안내 메시지
# 출력: " 정확한 정보는 인사팀(내선 1234)으로"
# flush=True: 출력 버퍼를 즉시 비워서 실시간 표시

# [청크 5]: 마무리
# 출력: " 문의해주세요."
# 최종: 모든 청크가 연결되어 완전한 문장 형성

# [스트리밍 출력의 시각적 효과]
# 사용자 화면에서 보이는 실제 출력:
#
# AI 답변 생성 중...
#
# FAQ에 입사 3년차 혜택과 (약 0.2초 후)
# 도서 구입 지원에 대한 (약 0.4초 후)
# 구체적인 정보가 없습니다. (약 0.6초 후)
# 정확한 정보는 인사팀(내선 1234)으로 (약 0.8초 후)
# 문의해주세요. (약 1.0초 후)
#
# ==============================
# [전체 답변]
# ==============================

# [스트리밍의 기술적 구현]
# 1. OpenAI 스트리밍 API: Server-Sent Events(SSE) 사용
# 2. 데이터 형식: 각 청크는 JSON 형식의 delta 객체
# 3. 예시 청크 데이터:
#    {
#      "choices": [{
#        "delta": {
#          "content": "FAQ에 입사"
#        }
#      }]
#    }
# 4. LangChain 처리: delta.content 추출 → 문자열 청크 반환

# [스트리밍의 장점]
# 1. 사용자 경험 향상: 기다리는 시간 감소, 진행 상황 시각화
# 2. 대기 시간 최소화: 첫 번째 토큰 도착 시 즉시 출력 시작
# 3. 오류 조기 감지: 생성 중 문제 발생 시 즉시 인지 가능
# 4. 자원 효율성: 전체 응답 대기 없이 점진적 처리

# [스트리밍의 단점]
# 1. 구현 복잡성: 버퍼 관리, 연결 유지 필요
# 2. 네트워크 오버헤드: 작은 청크들로 인한 다중 요청
# 3. 토큰화 문제: 한글의 경우 토큰 경계에서 깨질 수 있음
# 4. 일관성: 중간에 생성된 내용이 최종 내용과 다를 수 있음

# [스트리밍의 실제 응용 시나리오]
# 1. 채팅 애플리케이션: 메시지가 점진적으로 표시되는 자연스러운 대화
# 2. 콘텐츠 생성 도구: 글쓰기 과정의 실시간 시각화
# 3. 교육 도구: AI의 사고 과정을 보여주는 데모
# 4. 대화형 인터페이스: 빠른 첫 응답으로 대화 흐름 유지

# [스트리밍 파라미터 설정]
# LangChain에서 스트리밍 제어 방법:
# 1. stream() 메서드: 기본 스트리밍 활성화
# 2. batch() 메서드: 여러 질문 배치 처리
# 3. astream_events(): 이벤트 기반 스트리밍 (상세 제어)

# [실시간 출력의 사용자 경험 측면]
# 1. 심리적 효과: 기다림의 지루함 감소
# 2. 참여도 증가: 생성 과정에 대한 흥미 유발
# 3. 중단 가능성: 원하는 시점에 생성 중단 가능
# 4. 피드백: 생성 방향이 잘못되면 즉시 인지 가능

# [스트리밍과 invoke()의 성능 비교]
# 1. 총 소요 시간: 거의 동일 (스트리밍이 약간 더 빠를 수 있음)
# 2. 첫 바이트 시간: 스트리밍이 훨씬 빠름 (TTFB 감소)
# 3. 메모리 사용량: 스트리밍이 더 효율적 (청크 단위 처리)
# 4. 네트워크 사용량: 스트리밍이 약간 더 많음 (헤더 오버헤드)

# [스트리밍의 구현 난이도]
# 1. 기본 스트리밍: 간단 (rag_chain.stream()만 호출)
# 2. 고급 스트리밍: 복잡 (이벤트 처리, 오류 복구, 속도 제어)
# 3. 웹 인터페이스와 통합: 추가 작업 필요 (웹소켓, SSE 구현)

# [실제 서비스에서의 스트리밍 고려사항]
# 1. 네트워크 불안정: 연결 끊김 시 재연결 로직 필요
# 2. 보안: 스트리밍 연결의 보안 유지
# 3. 로깅: 스트리밍 세션의 로깅 방법
# 4. 모니터링: 스트리밍 성능 모니터링 지표

# [이 코드의 출력 결과 예시]
#
#  질문: 입사한 지 3년 되면 어떤 혜택이 있어? 그리고 책 사는 건 얼마나 지원해줘?
# AI 답변 생성 중...
#
# FAQ에 입사 3년차 혜택과 도서 구입 지원에 대한 구체적인 정보가 없습니다.
# 정확한 정보는 인사팀(내선 1234)으로 문의해주세요.
#
# ==============================
# FAQ에 입사 3년차 혜택과 도서 구입 지원에 대한 구체적인 정보가 없습니다.
# 정확한 정보는 인사팀(내선 1234)으로 문의해주세요.
# ==============================

# [스트리밍의 교육적 가치]
# 1. AI 작동 원리 이해: 토큰 단위 생성 과정 시각화
# 2. 프로그래밍 학습: 비동기 처리, 제너레이터 개념 실습
# 3. 사용자 인터페이스: 실시간 상호작용 디자인 학습
# 4. 시스템 설계: 점진적 처리와 즉시 응답의 트레이드오프 이해

# [스트리밍의 한계와 해결책]
# 1. 토큰화 문제: 한글의 경우 의미 단위로 잘리지 않을 수 있음
#   → 해결: 문장 단위 버퍼링 또는 의미 단위 청킹
# 2. 네트워크 지연: 느린 네트워크에서 끊김 현상
#   → 해결: 재시도 로직, 로컬 버퍼링
# 3. 일관성 문제: 중간 생성 내용 변경 가능성
#   → 해결: 생성 완료 후 최종 검증

# [스트리밍을 사용한 고급 기능]
# 1. 생성 속도 제어: typewriter 효과 (글자별 지연 출력)
# 2. 생성 중단: 사용자가 중간에 멈출 수 있는 기능
# 3. 실시간 수정: 생성 중 내용 수정 기능
# 4. 병렬 생성: 여러 응답 동시 생성 및 선택

# [최종 출력의 의미]
# print("\n" + "="*30): 구분선 출력
# print(response): invoke()로 생성된 전체 응답 출력 (비교용)
# stream()과 invoke()의 결과가 동일함을 확인하는 용도

# [스트리밍의 실제 적용 예시]
# 1. ChatGPT: 사용자가 질문하면 점진적으로 답변 표시
# 2. 코드 생성 도구: 코드가 줄 단위로 생성되어 표시
# 3. 번역 서비스: 문장 단위로 실시간 번역 결과 제공
# 4. 요약 도구: 요약문이 점차 완성되어 표시

# 이 코드는 단순한 기능 테스트를 넘어,
# 실시간 상호작용이 중요한 현대적 AI 애플리케이션의 구현 원리를 보여줍니다.

### 마무리 정리

In [ ]:
# 실습 종료 후 임시 파일 삭제
os.remove("company_policy.txt")